# SmartSpend — Transaction Categorisation Model

Trains and evaluates the TF-IDF + Logistic Regression expense categorisation model
as specified in Section 3.6.1 of the SmartSpend research proposal.

**Algorithm:** TF-IDF vectorisation (unigrams + bigrams, `max_features=5000`) + Logistic Regression  
**Input:** Transaction `description` text extracted from MTN MoMo SMS  
**Output:** One of 10 fixed Rwanda-context expense categories  
**Training data:** Synthetic MTN MoMo SMS dataset (~2,900 records)

Trained model artefacts are saved to `../backend_api/storage/models/` for direct use
by the FastAPI backend (`app/services/model_service.py`).

In [ ]:
import json
import os

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

In [ ]:
# Paths — dataset lives in backend_api/data/; model saves to backend_api/storage/models/
SMS_DATASET  = os.path.join("..", "backend_api", "data",
                             "smartspend_initial_synthetic_momo_sms_dataset.csv")
MODEL_OUT_DIR = os.path.join("..", "backend_api", "storage", "models")
os.makedirs(MODEL_OUT_DIR, exist_ok=True)

print("Dataset    :", SMS_DATASET)
print("Model dir  :", MODEL_OUT_DIR)

## 1. Load Dataset

In [ ]:
sms_df = pd.read_csv(SMS_DATASET)
print("Shape   :", sms_df.shape)
print("Columns :", list(sms_df.columns))
display(sms_df.head())

## 2. Data Quality Checks

In [ ]:
print("Missing values:")
display(sms_df.isna().sum().to_frame("missing_count"))

print("\nDuplicate rows:", sms_df.duplicated().sum())

print("\nTransaction type distribution:")
display(sms_df["transaction_type"].value_counts())

print("\nCategory distribution (SENT only):")
display(
    sms_df[sms_df["transaction_type"] == "SENT"]["category"]
    .value_counts()
    .to_frame("count")
)

## 3. Data Engineering

Only `SENT` transactions are used. The 10 fixed expense categories apply to outgoing
expenses; `RECEIVED` transactions feed the income side of the dashboard and are not
passed through the expense categoriser.

**Train/inference alignment:** Training uses the `description` column only — matching
exactly what `model_service.categorize()` passes to `model.predict([description])`.
The previous notebook trained on `description + raw_sms` combined, creating a
mismatch between training input and serving input. This is corrected here.

In [ ]:
expense_df = sms_df[sms_df["transaction_type"] == "SENT"].copy()
expense_df["description"] = expense_df["description"].fillna("").str.strip()
expense_df["text_length"] = expense_df["description"].str.len()
expense_df["word_count"]  = expense_df["description"].str.split().str.len()

amount_col = "amount_rwf" if "amount_rwf" in expense_df.columns else "amount"

print("Training rows (SENT):", len(expense_df))
display(
    expense_df[["description", amount_col, "category", "text_length", "word_count"]]
    .head(10)
)

## 4. Visualisations

In [ ]:
# 4.1 Transaction type distribution  /  4.2 Category distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sms_df["transaction_type"].value_counts().plot(kind="bar", ax=axes[0])
axes[0].set_title("Transaction Type Distribution")
axes[0].set_xlabel("Transaction Type")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

cat_counts = expense_df["category"].value_counts().sort_values(ascending=False)
cat_counts.plot(kind="bar", ax=axes[1])
axes[1].set_title("Expense Category Distribution — Synthetic Dataset")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# 4.3 Category percentage table
cat_pct = (cat_counts / cat_counts.sum() * 100).round(2)
display(cat_pct.to_frame("percentage_of_training_set"))

In [ ]:
# 4.4–4.6 Amount distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

expense_df[amount_col].plot(kind="hist", bins=30, ax=axes[0])
axes[0].set_title("Expense Amount Distribution")
axes[0].set_xlabel("Amount (RWF)")
axes[0].set_ylabel("Frequency")

expense_df.boxplot(column=amount_col, by="category", ax=axes[1], rot=45)
axes[1].set_title("Amount by Category")
axes[1].set_xlabel("")
axes[1].set_ylabel("Amount (RWF)")
fig.suptitle("")

expense_df.groupby("category")[amount_col].mean() \
    .sort_values(ascending=False).plot(kind="bar", ax=axes[2])
axes[2].set_title("Average Amount by Category")
axes[2].set_xlabel("")
axes[2].set_ylabel("Average (RWF)")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 4.7–4.9 Text length and word count
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

expense_df["text_length"].plot(kind="hist", bins=30, ax=axes[0, 0])
axes[0, 0].set_title("Description Length Distribution")
axes[0, 0].set_xlabel("Character Count")

expense_df["word_count"].plot(kind="hist", bins=20, ax=axes[0, 1])
axes[0, 1].set_title("Word Count Distribution")
axes[0, 1].set_xlabel("Word Count")

expense_df.boxplot(column="word_count", by="category", ax=axes[1, 0], rot=45)
axes[1, 0].set_title("Word Count by Category")
axes[1, 0].set_xlabel("")
fig.suptitle("")

expense_df.groupby("category")["text_length"].mean() \
    .sort_values(ascending=False).plot(kind="bar", ax=axes[1, 1])
axes[1, 1].set_title("Average Description Length by Category")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 5. Model Architecture

The categorisation model is a scikit-learn `Pipeline` with two steps:

| Step | Component | Configuration |
|---|---|---|
| Feature extraction | `TfidfVectorizer` | `ngram_range=(1,2)`, `max_features=5000`, `min_df=1` |
| Classifier | `LogisticRegression` | `C=1.0`, `class_weight="balanced"`, `max_iter=1000`, `random_state=42` |

**Why Logistic Regression + TF-IDF:** Short, domain-specific text (15–40 characters)
at a small dataset size. LR outperforms more complex models on this profile, retrains
fast per user correction cycle, and provides calibrated class probabilities used as
confidence scores. (See proposal Table 3.3 for the full algorithm comparison.)

**Why `class_weight="balanced"`:** The 10 categories are not perfectly equal in size.
Balanced weighting prevents the classifier from being biased toward the more frequent
categories (Transport, Personal Transfer).

## 6. Train and Evaluate

In [ ]:
# 6.1 Stratified 5-fold cross-validation (as specified in proposal Section 3.5)
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=5000, min_df=1)),
    ("classifier", LogisticRegression(
        C=1.0, max_iter=1000, class_weight="balanced", random_state=42
    )),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    pipeline,
    expense_df["description"],
    expense_df["category"],
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)
print(f"5-fold CV accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Fold scores        : {[round(s, 4) for s in cv_scores]}")

# 6.2 Hold-out test evaluation (80 / 20 stratified split)
X_train, X_test, y_train, y_test = train_test_split(
    expense_df["description"],
    expense_df["category"],
    test_size=0.2,
    random_state=42,
    stratify=expense_df["category"],
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

cat_metrics = {
    "test_accuracy":          float(accuracy_score(y_test, y_pred)),
    "test_precision_macro":   float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
    "test_recall_macro":      float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
    "test_f1_macro":          float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
    "test_f1_weighted":       float(f1_score(y_test, y_pred, average="weighted", zero_division=0)),
    "cv_mean_accuracy":       float(cv_scores.mean()),
    "cv_std_accuracy":        float(cv_scores.std()),
}
print(json.dumps(cat_metrics, indent=2))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# 6.3 Confusion matrices (raw + normalised)
labels = sorted(expense_df["category"].unique())
fig, axes = plt.subplots(1, 2, figsize=(22, 9))

cm = confusion_matrix(y_test, y_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(
    ax=axes[0], xticks_rotation=45, values_format="d"
)
axes[0].set_title("Confusion Matrix")

cm_norm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")
ConfusionMatrixDisplay(cm_norm, display_labels=labels).plot(
    ax=axes[1], xticks_rotation=45, values_format=".2f"
)
axes[1].set_title("Normalised Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
# 6.4 Per-category F1 scores
report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
per_class_df = pd.DataFrame(
    [{**v, "category": k} for k, v in report_dict.items() if k in labels]
).set_index("category")
display(per_class_df[["precision", "recall", "f1-score", "support"]])

plt.figure(figsize=(12, 5))
per_class_df["f1-score"].sort_values().plot(kind="bar")
plt.title("F1 Score by Expense Category")
plt.xlabel("Category")
plt.ylabel("F1 Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 6.5 Prediction confidence distribution
proba      = pipeline.predict_proba(X_test)
confidence = proba.max(axis=1)

confidence_df = pd.DataFrame({
    "true_category":      y_test.values,
    "predicted_category": y_pred,
    "confidence":         confidence,
    "correct":            y_test.values == y_pred,
})
display(confidence_df.head())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

confidence_df["confidence"].plot(kind="hist", bins=30, ax=axes[0])
axes[0].set_title("Confidence Score Distribution")
axes[0].set_xlabel("Max Predicted Probability")
axes[0].set_ylabel("Frequency")

confidence_df.boxplot(column="confidence", by="correct", ax=axes[1])
axes[1].set_title("Confidence by Prediction Correctness")
axes[1].set_xlabel("Prediction Correct")
axes[1].set_ylabel("Confidence")
plt.suptitle("")

plt.tight_layout()
plt.show()

In [ ]:
# 6.6 Top TF-IDF terms per category (from Logistic Regression coefficients)
tfidf_step    = pipeline.named_steps["tfidf"]
clf_step      = pipeline.named_steps["classifier"]
feature_names = np.array(tfidf_step.get_feature_names_out())

top_terms_rows = []
for class_idx, class_label in enumerate(clf_step.classes_):
    top_indices = np.argsort(clf_step.coef_[class_idx])[-10:][::-1]
    top_terms_rows.append({
        "category":  class_label,
        "top_terms": ", ".join(feature_names[top_indices]),
    })

display(pd.DataFrame(top_terms_rows).set_index("category"))

## 7. Save Model Artefacts

The trained pipeline is saved to `../backend_api/storage/models/` where it is
loaded by `app/services/model_service.py` on startup.

Note: The model saved here is trained on the 80% training split (for honest
evaluation). The backend retraining endpoint (`POST /models/category/retrain`)
trains on the full dataset after cross-validation completes.

In [ ]:
model_path = os.path.join(MODEL_OUT_DIR, "smartspend_category_model.joblib")
joblib.dump(pipeline, model_path)
print("Model saved   :", model_path)

metrics_path = os.path.join(MODEL_OUT_DIR, "categorisation_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(cat_metrics, f, indent=2)
print("Metrics saved :", metrics_path)